In [21]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [11]:
# Install required packages
!pip install -q transformers diffusers accelerate torchaudio pillow datasets
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q torchcodec

print("✓ All packages installed!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 119.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 875.6/875.6 kB 65.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 144.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 663.9/663.9 MB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.9/417.9 MB 6.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.4/168.4 MB 15.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 MB 43.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 128.2/128.2 MB 19.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.1/204.1 MB 12.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 MB 7.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.1/99.1 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 905.2/905.2 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━

In [22]:
import os

# ====== UPDATE THESE PATHS ======
# Path to your folder in Google Drive containing the files
DRIVE_FOLDER = "/content/drive/MyDrive/Audio2Image"  # Change this to your folder path

# File names (update if different)
MAIN_PY_FILE = "main2.py"
CSV_FILE = "main_dataV1.csv"
ZIP_FOLDER = DRIVE_FOLDER  # Folder containing vggsound_00.zip and vggsound_01.zip

# Verify files exist
print("Checking files...")
print(f"Drive folder: {DRIVE_FOLDER}")
print(f"  Exists: {os.path.exists(DRIVE_FOLDER)}")

main_py_path = os.path.join(DRIVE_FOLDER, MAIN_PY_FILE)
csv_path = os.path.join(DRIVE_FOLDER, CSV_FILE)

print(f"\nFiles:")
print(f"  main2.py: {os.path.exists(main_py_path)} - {main_py_path}")
print(f"  CSV: {os.path.exists(csv_path)} - {csv_path}")

# Check for ZIP files
print(f"\nZIP files in {ZIP_FOLDER}:")
for f in os.listdir(ZIP_FOLDER):
    if f.endswith('.zip'):
        print(f"  ✓ {f}")

Checking files...
Drive folder: /content/drive/MyDrive/Audio2Image
  Exists: True

Files:
  main2.py: True - /content/drive/MyDrive/Audio2Image/main2.py
  CSV: True - /content/drive/MyDrive/Audio2Image/main_dataV1.csv

ZIP files in /content/drive/MyDrive/Audio2Image:
  ✓ vggsound_04.zip
  ✓ vggsound_00.zip
  ✓ vggsound_02.zip


In [23]:
import shutil

# Copy main2.py to current directory
if os.path.exists(main_py_path):
    shutil.copy(main_py_path, "/content/main2.py")
    print("✓ main2.py copied to /content/")
else:
    print(f"❌ ERROR: main2.py not found at {main_py_path}")
    print("Please update DRIVE_FOLDER path in the previous cell")

✓ main2.py copied to /content/


In [24]:
# Change to content directory
os.chdir("/content")

# Import the training module
import sys
sys.path.insert(0, '/content')

from main2 import Config, train, infer

print("✓ Training module imported successfully!")

✓ Training module imported successfully!


In [25]:
# Create configuration
cfg = Config()

# Update paths to use Google Drive
cfg.train_csv = csv_path
cfg.image_folder = ZIP_FOLDER
cfg.use_zip_files = True

# Save checkpoint to Drive (so it persists after Colab session)
cfg.ckpt_path = os.path.join(DRIVE_FOLDER, "audio2image_mapper_dual.pt")

# Training settings (adjust as needed)
cfg.batch_size = 4  # Increase if you have enough GPU memory
cfg.max_epochs = 10  # Start with fewer epochs for testing
cfg.lr = 2e-4

# Multi-task loss weights
cfg.clap_loss_weight = 0.5
cfg.sd_loss_weight = 1.0
cfg.diffusion_loss_weight = 1.0

# Fine-tuning settings
cfg.finetune_sd = True  # Set to False for faster training (mapper only)
cfg.freeze_vae = True
cfg.freeze_text_encoder = True

# Evaluation settings
cfg.eval_every_n_epochs = 2  # Evaluate every 2 epochs
cfg.num_eval_samples = 2  # Number of samples to generate during eval
cfg.save_eval_images = True

# Print configuration
print("Training Configuration:")
print(f"  Device: {cfg.device}")
print(f"  Batch size: {cfg.batch_size}")
print(f"  Epochs: {cfg.max_epochs}")
print(f"  Learning rate: {cfg.lr}")
print(f"  Fine-tune SD: {cfg.finetune_sd}")
print(f"  CSV: {cfg.train_csv}")
print(f"  Image folder: {cfg.image_folder}")
print(f"  Checkpoint: {cfg.ckpt_path}")

Training Configuration:
  Device: cuda
  Batch size: 4
  Epochs: 10
  Learning rate: 0.0002
  Fine-tune SD: True
  CSV: /content/drive/MyDrive/Audio2Image/main_dataV1.csv
  Image folder: /content/drive/MyDrive/Audio2Image
  Checkpoint: /content/drive/MyDrive/Audio2Image/audio2image_mapper_dual.pt


In [26]:
import torch

print("GPU Information:")
print(f"  CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  GPU name: {torch.cuda.get_device_name(0)}")
    print(f"  GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print(f"  Current device: {cfg.device}")
else:
    print("  ⚠️ WARNING: No GPU detected! Training will be very slow.")
    print("  Go to Runtime > Change runtime type > Hardware accelerator > GPU")

GPU Information:
  CUDA available: True
  GPU name: NVIDIA A100-SXM4-80GB
  GPU memory: 85.17 GB
  Current device: cuda


In [27]:
from main2 import AudioCaptionDataset

print("Testing dataset loading...")
try:
    test_ds = AudioCaptionDataset(
        cfg.train_csv,
        cfg.image_folder,
        use_zip_files=cfg.use_zip_files
    )
    print(f"\n✓ Dataset loaded successfully!")
    print(f"  Total samples: {len(test_ds)}")

    # Test loading first sample
    print("\nTesting first sample...")
    wav, sr, caption, img, has_img = test_ds[0]
    print(f"  Audio shape: {wav.shape}")
    print(f"  Sample rate: {sr}")
    print(f"  Caption: {caption}")
    print(f"  Image shape: {img.shape}")
    print(f"  Has image: {has_img}")
    print("\n✓ Dataset test passed!")

except Exception as e:
    print(f"\n❌ Dataset loading failed: {e}")
    print("\nPlease check:")
    print("  1. CSV file path is correct")
    print("  2. ZIP files (vggsound_00.zip, vggsound_01.zip) are in the image_folder")
    print("  3. CSV format matches expected structure")

Testing dataset loading...
Loading dataset from: /content/drive/MyDrive/Audio2Image/main_dataV1.csv
Base folder: /content/drive/MyDrive/Audio2Image
Use ZIP files: True
Searching for ZIP files...
  ✓ Opened vggsound_04.zip (key: 'vggsound_04', 7523 files)
  ✓ Opened vggsound_00.zip (key: 'vggsound_00', 20003 files)
  ✓ Opened vggsound_02.zip (key: 'vggsound_02', 17334 files)
Row 1: base_folder='vggsound_00', audio='vggsound_00/audio/g-f_I2yQ_000001.wav', exists=True
Row 2: base_folder='vggsound_00', audio='vggsound_00/audio/0PQM4-hqg_000030.wav', exists=True
Row 3: base_folder='vggsound_00', audio='vggsound_00/audio/56QUhyDQM_000185.wav', exists=True
✓ Loaded 18643 audio files (18643 with matching images)

✓ Dataset loaded successfully!
  Total samples: 18643

Testing first sample...
  Audio shape: torch.Size([482400])
  Sample rate: 48000
  Caption: people marching
  Image shape: torch.Size([3, 512, 512])
  Has image: True

✓ Dataset test passed!


In [28]:
# Start training
print("Starting training...\n")
train(cfg)
print("\n✅ Training complete!")

Starting training...

Loading dataset from: /content/drive/MyDrive/Audio2Image/main_dataV1.csv
Base folder: /content/drive/MyDrive/Audio2Image
Use ZIP files: True
Searching for ZIP files...
  ✓ Opened vggsound_04.zip (key: 'vggsound_04', 7523 files)
  ✓ Opened vggsound_00.zip (key: 'vggsound_00', 20003 files)
  ✓ Opened vggsound_02.zip (key: 'vggsound_02', 17334 files)
Row 1: base_folder='vggsound_00', audio='vggsound_00/audio/g-f_I2yQ_000001.wav', exists=True
Row 2: base_folder='vggsound_00', audio='vggsound_00/audio/0PQM4-hqg_000030.wav', exists=True
Row 3: base_folder='vggsound_00', audio='vggsound_00/audio/56QUhyDQM_000185.wav', exists=True
✓ Loaded 18643 audio files (18643 with matching images)

Dataset split:
  Training: 16778 samples
  Validation: 1865 samples

Loading CLAP model...
Loading CLIP for evaluation...
  ✓ CLIP loaded (frozen for evaluation only)
Loading Stable Diffusion...


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

🔥 End-to-End Training Mode:
  ✓ UNet: TRAINABLE
  ✓ VAE: FROZEN
  ✓ Text Encoder: FROZEN
Creating MLP: CLAP audio (512) → CLAP text (512) & SD (768)

🔥 Setting up END-TO-END training:
  Mapper optimizer: LR=0.0002
  SD UNet optimizer: LR=1e-05

Starting End-to-End Training
Dataset: 18643 samples (16778 train, 1865 val)
Batch size: 4
Epochs: 10
Evaluation: Every 2 epoch(s)
Loss weights:
  CLAP: 0.5
  SD Embedding: 1.0
  Diffusion: 1.0



Epoch 1/10 [TRAIN]: 100%|██████████| 4194/4194 [1:14:05<00:00,  1.06s/it, total loss=1.207, diff=0.107, c_sim=0.36, s_sim=0.68]



Epoch 1 Summary:
  Total Loss: 0.8739
  CLAP Loss: 0.3498 | Sim: 0.341
  SD Loss: 0.5617 | Sim: 0.683
  Diffusion Loss: 0.1372

💾 Checkpoint saved: /content/drive/MyDrive/Audio2Image/audio2image_mapper_dual.pt



...

  0%|          | 0/50 [00:00<?, ?it/s]

  Batch 2/3: Avg CLIP = 21.809


  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

  Batch 3/3: Avg CLIP = 18.886
  Sample 0: 'printer printing...' | CLIP: 18.802
    Saved to: eval_samples/ep10_sample0_score18.80.png
  Sample 1: 'rapping...' | CLIP: 19.437
    Saved to: eval_samples/ep10_sample1_score19.44.png
  Sample 2: 'roller coaster running...' | CLIP: 20.130
    Saved to: eval_samples/ep10_sample2_score20.13.png
  Sample 3: 'female singing...' | CLIP: 23.488
    Saved to: eval_samples/ep10_sample3_score23.49.png

📊 Epoch 10 Summary:
Training Metrics:
  Total Loss: 0.4965
  CLAP Loss: 0.1004 | Sim: 0.462
  SD Loss: 0.3100 | Sim: 0.839
  Diffusion Loss: 0.1363

Validation Metrics:
  🎯 CLIP Score: 19.938 (higher = better image-text match)

💾 Checkpoint saved: /content/drive/MyDrive/Audio2Image/audio2image_mapper_dual.pt
   Current best CLIP: 22.311

🎉 Training completed!
   Best CLIP score achieved: 22.311

✅ Training complete!
